# Objective 3 — Step 10: Frozen Sigmoid Calibration Replication

Step 9 selected **Sigmoid / Platt-style calibration** on German Credit using German evidence only.

This notebook freezes that calibration rule and independently applies it to:
- Australian Credit Approval
- Taiwan Credit Card Default

The core hybrid remains unchanged:
**Group-aware Chi-Square Top-75% → Balanced LR + Balanced RF + Balanced XGBoost → Equal-probability Soft Voting**

For every outer fold, the sigmoid calibrator is fitted only from 5-fold grouped inner out-of-fold hybrid probabilities from the outer-training partition. The outer-test fold is untouched until final evaluation.

The classification threshold remains **0.50**. Threshold optimisation is deferred to the next stage.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, log_loss,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")
STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP8_DIR = BASE_DIR / "results" / "frozen_hybrid_ensemble_replication"
STEP9_DIR = BASE_DIR / "results" / "calibration_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "frozen_sigmoid_calibration_replication"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP8_PREDICTIONS_FILE = STEP8_DIR / "frozen_hybrid_outer_predictions.csv"
STEP9_SUMMARY_FILE = STEP9_DIR / "german_calibration_summary.csv"

DATASET_FILES = {
    "Australian Credit Approval": DATA_DIR / "australian_credit_approval_cleaned.csv",
    "Taiwan Credit Card Default": DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

required = [
    DICTIONARY_FILE, BASELINE_PREDICTIONS_FILE, STEP8_PREDICTIONS_FILE,
    STEP9_SUMMARY_FILE, *DATASET_FILES.values()
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing previous-step files:\n" + "\n".join(missing))

REPEAT_SEEDS = [42, 142, 242, 342, 442]
OUTER_FOLDS = 5
INNER_FOLDS = 5
FROZEN_FRACTION = 0.75
BASE_MODELS = ["LR", "RF", "XGB"]
EPS = 1e-6

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_sigmoid_calibration_replication


## 1. Verify the Step-9 German decision

In [3]:

step9_summary = pd.read_csv(STEP9_SUMMARY_FILE)
display(step9_summary.sort_values(["Brier", "LogLoss", "ECE"]))

sigmoid_row = step9_summary[
    step9_summary["calibration_method"] == "Sigmoid"
].copy()

if len(sigmoid_row) != 1:
    raise RuntimeError("Sigmoid result was not uniquely identified.")

frozen_rule = {
    "development_dataset": "German Credit",
    "calibration_method": "Sigmoid / Platt-style logistic recalibration",
    "fit_input": "logit of inner OOF raw hybrid probability",
    "external_method_selection": False,
    "threshold": 0.50,
}

with open(
    OUT_DIR / "frozen_sigmoid_calibration_rule.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(frozen_rule, f, indent=4)

print(json.dumps(frozen_rule, indent=2))


,calibration_method,Brier,Brier_SD,LogLoss,LogLoss_SD,ECE,Calibration_Intercept,Calibration_Slope,Mean_Predicted_Probability,Observed_Adverse_Rate,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC
1,Sigmoid,0.160564,0.011548,0.487191,0.030339,0.073567,0.031212,1.096170,0.302790,0.3,0.798802,0.634897,0.464135,0.646891,0.536208,0.677750,0.395607
0,Isotonic,0.162829,0.012362,0.501964,0.042407,0.066442,-0.082957,0.907247,0.302656,0.3,0.792547,0.596490,0.475006,0.643966,0.539162,0.679963,0.397587
2,Uncalibrated,0.165559,0.010302,0.498661,0.026643,0.091794,-0.396548,1.004470,0.363889,0.3,0.798802,0.634897,0.626506,0.586505,0.603075,0.718355,0.428590


{
  "development_dataset": "German Credit",
  "calibration_method": "Sigmoid / Platt-style logistic recalibration",
  "fit_input": "logit of inner OOF raw hybrid probability",
  "external_method_selection": false,
  "threshold": 0.5
}


## 2. Load datasets and feature roles

In [4]:

dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)
step8_predictions = pd.read_csv(STEP8_PREDICTIONS_FILE)

datasets = {name: pd.read_csv(path) for name, path in DATASET_FILES.items()}

def roles_for(dataset_name):
    d = dictionary[dictionary["dataset"] == dataset_name].copy()
    return {
        role: d.loc[d["role"] == role, "variable"].astype(str).tolist()
        for role in ["categorical", "ordinal", "numerical", "identifier"]
    }

roles = {name: roles_for(name) for name in datasets}

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]
    print(
        dataset_name,
        "| N =", len(df),
        "| predictors =", len(predictors),
        "| Top75 =", math.ceil(len(predictors) * FROZEN_FRACTION),
        "| adverse rate =", round(df["adverse_target"].mean(), 4),
    )


Australian Credit Approval | N = 690 | predictors = 14 | Top75 = 11 | adverse rate = 0.5551
Taiwan Credit Card Default | N = 30000 | predictors = 23 | Top75 = 18 | adverse rate = 0.2212


## 3. Preprocessing and frozen feature selection

In [5]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)


def build_preprocessor(dataset_name, mode, selected_features):
    r = roles[dataset_name]
    selected_features = list(selected_features)

    selected_cat = [f for f in r["categorical"] if f in selected_features]
    selected_ord = [f for f in r["ordinal"] if f in selected_features]
    selected_num = [f for f in r["numerical"] if f in selected_features]

    transformers = []

    if selected_num:
        if mode == "scaled":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        elif mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)
        transformers.append(("num", num_pipe, selected_num))

    if selected_ord:
        if mode == "scaled":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        elif mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)
        transformers.append(("ord", ord_pipe, selected_ord))

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", cat_pipe, selected_cat))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


def feature_map_from_fitted_chi2_preprocessor(prep, dataset_name):
    r = roles[dataset_name]
    rows = []
    idx = 0

    for feature in r["numerical"]:
        rows.append({"transformed_index": idx, "source_feature": feature, "source_role": "numerical"})
        idx += 1

    for feature in r["ordinal"]:
        rows.append({"transformed_index": idx, "source_feature": feature, "source_role": "ordinal"})
        idx += 1

    if r["categorical"]:
        encoder = prep.named_transformers_["cat"].named_steps["onehot"]
        for source_feature, categories in zip(r["categorical"], encoder.categories_):
            for _ in categories:
                rows.append({"transformed_index": idx, "source_feature": source_feature, "source_role": "categorical"})
                idx += 1

    fmap = pd.DataFrame(rows)
    assert len(fmap) == len(prep.get_feature_names_out())
    return fmap


def group_aware_chi2_top75(dataset_name, X_train, y_train):
    r = roles[dataset_name]
    all_features = r["categorical"] + r["ordinal"] + r["numerical"]

    prep = build_preprocessor(dataset_name, "chi2", all_features)
    X_chi = prep.fit_transform(X_train[all_features], y_train)
    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_chi2_preprocessor(prep, dataset_name)
    raw_scores, _ = chi2(X_chi, y_train)

    temp = fmap.copy()
    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)
    ranks = temp["raw_score"].rank(ascending=False, method="average")
    temp["normalized_relevance"] = (
        1.0 - (ranks - 1.0) / (n - 1.0) if n > 1 else 1.0
    )

    grouped = (
        temp.groupby("source_feature", as_index=False)
        .agg(group_score=("normalized_relevance", "mean"))
    )

    grouped["source_rank"] = grouped["group_score"].rank(
        ascending=False, method="average"
    )

    grouped = grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)

    n_select = int(math.ceil(len(grouped) * FROZEN_FRACTION))
    return grouped["source_feature"].astype(str).tolist()[:n_select]


## 4. Frozen balanced hybrid

In [6]:

def balanced_ratio(y_train):
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    return n_neg / n_pos


def build_balanced_base_pipeline(dataset_name, model_name, selected_features, y_train):
    if model_name == "LR":
        estimator = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42,
        )
        mode = "scaled"

    elif model_name == "RF":
        estimator = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        mode = "tree"

    elif model_name == "XGB":
        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(balanced_ratio(y_train)),
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
        mode = "tree"

    else:
        raise ValueError(model_name)

    return Pipeline([
        ("preprocessor", build_preprocessor(dataset_name, mode, selected_features)),
        ("model", estimator),
    ])


def fit_hybrid_and_predict(dataset_name, X_train, y_train, X_predict, selected_features):
    probabilities = []

    for model_name in BASE_MODELS:
        pipe = build_balanced_base_pipeline(
            dataset_name,
            model_name,
            selected_features,
            y_train,
        )
        pipe.fit(X_train[selected_features], y_train)
        probabilities.append(
            pipe.predict_proba(X_predict[selected_features])[:, 1]
        )

    return np.column_stack(probabilities).mean(axis=1)


## 5. Frozen sigmoid calibrator and metrics

In [7]:

def clip_probability(p):
    return np.clip(np.asarray(p, dtype=float), EPS, 1.0 - EPS)


def logit(p):
    p = clip_probability(p)
    return np.log(p / (1.0 - p))


def fit_sigmoid_calibrator(raw_probability, y_true):
    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )
    model.fit(logit(raw_probability).reshape(-1, 1), y_true)
    return model


def apply_sigmoid_calibrator(model, raw_probability):
    return model.predict_proba(
        logit(raw_probability).reshape(-1, 1)
    )[:, 1]


def expected_calibration_error(y_true, probability, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probability, edges[1:-1], right=True)

    ece = 0.0
    total = len(y_true)

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        n = int(mask.sum())
        if n == 0:
            continue

        observed = y_true[mask].mean()
        predicted = probability[mask].mean()
        ece += (n / total) * abs(observed - predicted)

    return float(ece)


def calibration_intercept_slope(y_true, probability):
    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )
    model.fit(logit(probability).reshape(-1, 1), y_true)

    return float(model.intercept_[0]), float(model.coef_[0, 0])


def calculate_metrics(y_true, probability, threshold=0.50):
    probability = clip_probability(probability)
    prediction = (probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, prediction, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    gmean = (
        math.sqrt(specificity * sensitivity)
        if not np.isnan(specificity + sensitivity)
        else np.nan
    )

    fpr, tpr, _ = roc_curve(y_true, probability)
    ks = float(np.max(tpr - fpr))

    intercept, slope = calibration_intercept_slope(y_true, probability)

    return {
        "brier_score": brier_score_loss(y_true, probability),
        "log_loss": log_loss(y_true, probability, labels=[0, 1]),
        "ece_10bin": expected_calibration_error(y_true, probability, 10),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
        "mean_predicted_probability": float(np.mean(probability)),
        "observed_adverse_rate": float(np.mean(y_true)),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision_adverse": precision_score(
            y_true, prediction, pos_label=1, zero_division=0
        ),
        "recall_adverse": recall_score(
            y_true, prediction, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true, prediction, pos_label=1, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "mcc": matthews_corrcoef(y_true, prediction),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 6. Recreate and verify the exact outer folds

In [8]:

def build_verified_splits(dataset_name, df):
    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    X_local = df[predictors].copy()
    y_local = df["adverse_target"].astype(int).copy()
    groups_local = df["profile_group_id"].astype(str).copy()

    saved = baseline_predictions[
        (baseline_predictions["dataset"] == dataset_name)
        & (baseline_predictions["model"] == "XGB")
    ].copy()

    split_dict = {}

    for repeat_no, seed in enumerate(REPEAT_SEEDS, start=1):
        splitter = StratifiedGroupKFold(
            n_splits=OUTER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (train_idx, test_idx) in enumerate(
            splitter.split(X_local, y_local, groups_local),
            start=1,
        ):
            run_id = f"R{repeat_no}_F{fold_no}"

            expected_test = set(np.asarray(test_idx, dtype=int).tolist())
            saved_test = set(
                saved.loc[
                    saved["run_id"] == run_id,
                    "source_row_index",
                ].astype(int).tolist()
            )

            assert expected_test == saved_test

            assert len(
                set(groups_local.iloc[train_idx]).intersection(
                    set(groups_local.iloc[test_idx])
                )
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(train_idx, dtype=int),
                "test_idx": np.asarray(test_idx, dtype=int),
            }

    return split_dict


all_splits = {
    dataset_name: build_verified_splits(dataset_name, df)
    for dataset_name, df in datasets.items()
}

print("All replication folds exactly match Step 3.")


All replication folds exactly match Step 3.


## 7. Run frozen sigmoid calibration replication

This is the longest cell because fresh inner OOF hybrid probabilities are required for every outer fold.


In [9]:

fold_rows = []
prediction_rows = []
calibrator_rows = []
inner_oof_rows = []

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    predictors = r["categorical"] + r["ordinal"] + r["numerical"]

    X_local = df[predictors].copy()
    y_local = df["adverse_target"].astype(int).copy()
    groups_local = df["profile_group_id"].astype(str).copy()

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    for run_number, (run_id, info) in enumerate(
        all_splits[dataset_name].items(),
        start=1,
    ):
        train_idx = info["train_idx"]
        test_idx = info["test_idx"]

        X_outer_train = X_local.iloc[train_idx].copy()
        X_outer_test = X_local.iloc[test_idx].copy()
        y_outer_train = y_local.iloc[train_idx].copy()
        y_outer_test = y_local.iloc[test_idx].copy()
        g_outer_train = groups_local.iloc[train_idx].copy()

        oof_probability = np.full(
            len(X_outer_train),
            np.nan,
            dtype=float,
        )

        inner_seed = 30000 + info["seed"] + info["fold"]

        inner_splitter = StratifiedGroupKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=inner_seed,
        )

        for inner_fold, (inner_train_pos, inner_valid_pos) in enumerate(
            inner_splitter.split(
                X_outer_train,
                y_outer_train,
                g_outer_train,
            ),
            start=1,
        ):
            X_inner_train = X_outer_train.iloc[inner_train_pos].copy()
            X_inner_valid = X_outer_train.iloc[inner_valid_pos].copy()
            y_inner_train = y_outer_train.iloc[inner_train_pos].copy()

            selected_inner = group_aware_chi2_top75(
                dataset_name,
                X_inner_train,
                y_inner_train,
            )

            raw_inner_probability = fit_hybrid_and_predict(
                dataset_name,
                X_inner_train,
                y_inner_train,
                X_inner_valid,
                selected_inner,
            )

            oof_probability[inner_valid_pos] = raw_inner_probability

            for local_pos, outer_train_pos in enumerate(inner_valid_pos):
                inner_oof_rows.append({
                    "dataset": dataset_name,
                    "run_id": run_id,
                    "inner_fold": inner_fold,
                    "outer_train_position": int(outer_train_pos),
                    "y_true": int(y_outer_train.iloc[outer_train_pos]),
                    "raw_oof_hybrid_probability": float(
                        raw_inner_probability[local_pos]
                    ),
                })

        assert np.isfinite(oof_probability).all()

        sigmoid = fit_sigmoid_calibrator(
            oof_probability,
            y_outer_train,
        )

        calibrator_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "sigmoid_intercept": float(sigmoid.intercept_[0]),
            "sigmoid_slope": float(sigmoid.coef_[0, 0]),
            "outer_training_records": len(y_outer_train),
            "outer_training_adverse_rate": float(y_outer_train.mean()),
            "inner_oof_mean_raw_probability": float(
                np.mean(oof_probability)
            ),
        })

        selected_outer = group_aware_chi2_top75(
            dataset_name,
            X_outer_train,
            y_outer_train,
        )

        raw_outer_probability = fit_hybrid_and_predict(
            dataset_name,
            X_outer_train,
            y_outer_train,
            X_outer_test,
            selected_outer,
        )

        sigmoid_outer_probability = apply_sigmoid_calibrator(
            sigmoid,
            raw_outer_probability,
        )

        # Exact raw probability reproduction against Step 8.
        saved_step8 = (
            step8_predictions[
                (step8_predictions["dataset"] == dataset_name)
                & (step8_predictions["run_id"] == run_id)
            ]
            .sort_values("source_row_index")
        )

        current_order = np.argsort(test_idx)
        current_sorted = raw_outer_probability[current_order]
        saved_raw = saved_step8["hybrid_probability"].to_numpy(dtype=float)

        max_abs_diff = float(np.max(np.abs(current_sorted - saved_raw)))

        assert max_abs_diff < 1e-10, (
            f"{dataset_name} {run_id}: Step-8 reproduction failed; "
            f"max diff={max_abs_diff}"
        )

        probability_by_method = {
            "Uncalibrated": raw_outer_probability,
            "Sigmoid": sigmoid_outer_probability,
        }

        for method, probability in probability_by_method.items():
            metric_values = calculate_metrics(
                y_outer_test,
                probability,
                threshold=0.50,
            )

            fold_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": info["repeat"],
                "fold": info["fold"],
                "calibration_method": method,
                "selected_source_features": len(selected_outer),
                **metric_values,
            })

        for local_position, source_index in enumerate(test_idx):
            prediction_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": info["repeat"],
                "fold": info["fold"],
                "source_row_index": int(source_index),
                "y_true": int(y_outer_test.iloc[local_position]),
                "uncalibrated_probability": float(
                    raw_outer_probability[local_position]
                ),
                "sigmoid_probability": float(
                    sigmoid_outer_probability[local_position]
                ),
            })

        raw_row = [
            row for row in fold_rows
            if (
                row["dataset"] == dataset_name
                and row["run_id"] == run_id
                and row["calibration_method"] == "Uncalibrated"
            )
        ][-1]

        sigmoid_row_local = [
            row for row in fold_rows
            if (
                row["dataset"] == dataset_name
                and row["run_id"] == run_id
                and row["calibration_method"] == "Sigmoid"
            )
        ][-1]

        print(
            f"{run_id} ({run_number}/25) | "
            f"raw Brier={raw_row['brier_score']:.4f} | "
            f"sigmoid Brier={sigmoid_row_local['brier_score']:.4f} | "
            f"diff={max_abs_diff:.1e}"
        )


fold_results = pd.DataFrame(fold_rows)
predictions = pd.DataFrame(prediction_rows)
calibrators = pd.DataFrame(calibrator_rows)
inner_oof = pd.DataFrame(inner_oof_rows)

fold_results.to_csv(
    OUT_DIR / "frozen_sigmoid_replication_fold_results.csv",
    index=False,
)

predictions.to_csv(
    OUT_DIR / "frozen_sigmoid_replication_outer_predictions.csv",
    index=False,
)

calibrators.to_csv(
    OUT_DIR / "frozen_sigmoid_calibrator_parameters.csv",
    index=False,
)

inner_oof.to_csv(
    OUT_DIR / "frozen_sigmoid_inner_oof_probabilities.csv",
    index=False,
)

print("\nFrozen sigmoid calibration replication completed.")



Australian Credit Approval
R1_F1 (1/25) | raw Brier=0.1142 | sigmoid Brier=0.1131 | diff=1.1e-16
R1_F2 (2/25) | raw Brier=0.0896 | sigmoid Brier=0.0889 | diff=1.1e-16
R1_F3 (3/25) | raw Brier=0.0883 | sigmoid Brier=0.0882 | diff=1.1e-16
R1_F4 (4/25) | raw Brier=0.1087 | sigmoid Brier=0.1110 | diff=1.1e-16
R1_F5 (5/25) | raw Brier=0.0810 | sigmoid Brier=0.0806 | diff=1.1e-16
R2_F1 (6/25) | raw Brier=0.1173 | sigmoid Brier=0.1179 | diff=1.1e-16
R2_F2 (7/25) | raw Brier=0.1163 | sigmoid Brier=0.1161 | diff=1.1e-16
R2_F3 (8/25) | raw Brier=0.0921 | sigmoid Brier=0.0923 | diff=1.1e-16
R2_F4 (9/25) | raw Brier=0.1005 | sigmoid Brier=0.1005 | diff=1.1e-16
R2_F5 (10/25) | raw Brier=0.0596 | sigmoid Brier=0.0598 | diff=1.1e-16
R3_F1 (11/25) | raw Brier=0.1215 | sigmoid Brier=0.1213 | diff=1.1e-16
R3_F2 (12/25) | raw Brier=0.1062 | sigmoid Brier=0.1050 | diff=1.1e-16
R3_F3 (13/25) | raw Brier=0.0892 | sigmoid Brier=0.0896 | diff=1.1e-16
R3_F4 (14/25) | raw Brier=0.0860 | sigmoid Brier=0.0869 | 

## 8. Replication summary and paired deltas

In [10]:

summary = (
    fold_results
    .groupby(
        ["dataset", "calibration_method"],
        as_index=False,
    )
    .agg(
        Brier=("brier_score", "mean"),
        Brier_SD=("brier_score", "std"),
        LogLoss=("log_loss", "mean"),
        LogLoss_SD=("log_loss", "std"),
        ECE=("ece_10bin", "mean"),
        Calibration_Intercept=("calibration_intercept", "mean"),
        Calibration_Slope=("calibration_slope", "mean"),
        Mean_Predicted_Probability=("mean_predicted_probability", "mean"),
        Observed_Adverse_Rate=("observed_adverse_rate", "mean"),
        ROC_AUC=("roc_auc", "mean"),
        PR_AUC=("pr_auc", "mean"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=("balanced_accuracy", "mean"),
        MCC=("mcc", "mean"),
    )
)

summary.to_csv(
    OUT_DIR / "frozen_sigmoid_replication_summary.csv",
    index=False,
)

reference = (
    fold_results[
        fold_results["calibration_method"] == "Uncalibrated"
    ][
        [
            "dataset", "run_id",
            "brier_score", "log_loss", "ece_10bin",
            "roc_auc", "pr_auc", "recall_adverse",
            "f1_adverse", "balanced_accuracy", "mcc",
        ]
    ]
    .copy()
)

reference = reference.rename(
    columns={
        col: "reference_" + col
        for col in reference.columns
        if col not in ["dataset", "run_id"]
    }
)

paired = fold_results.merge(
    reference,
    on=["dataset", "run_id"],
    how="left",
    validate="many_to_one",
)

for metric in [
    "brier_score", "log_loss", "ece_10bin",
    "roc_auc", "pr_auc", "recall_adverse",
    "f1_adverse", "balanced_accuracy", "mcc",
]:
    paired["delta_" + metric] = (
        paired[metric] - paired["reference_" + metric]
    )

paired.to_csv(
    OUT_DIR / "frozen_sigmoid_replication_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired
    .groupby(
        ["dataset", "calibration_method"],
        as_index=False,
    )
    .agg(
        Delta_Brier=("delta_brier_score", "mean"),
        Delta_LogLoss=("delta_log_loss", "mean"),
        Delta_ECE=("delta_ece_10bin", "mean"),
        Delta_ROC_AUC=("delta_roc_auc", "mean"),
        Delta_PR_AUC=("delta_pr_auc", "mean"),
        Delta_Recall=("delta_recall_adverse", "mean"),
        Delta_F1=("delta_f1_adverse", "mean"),
        Delta_Balanced_Accuracy=("delta_balanced_accuracy", "mean"),
        Delta_MCC=("delta_mcc", "mean"),
    )
)

delta_summary.to_csv(
    OUT_DIR / "frozen_sigmoid_replication_delta_summary.csv",
    index=False,
)

display(summary)
display(delta_summary)


,dataset,calibration_method,Brier,Brier_SD,LogLoss,LogLoss_SD,ECE,Calibration_Intercept,Calibration_Slope,Mean_Predicted_Probability,Observed_Adverse_Rate,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC
0,Australian Credit Approval,Sigmoid,0.096920,0.014943,0.327660,0.043063,0.066162,0.014385,1.036799,0.553504,0.555072,0.931624,0.938131,0.864474,0.892598,0.877628,0.867758,0.732658
1,Australian Credit Approval,Uncalibrated,0.096957,0.015035,0.327467,0.042942,0.061645,0.099136,1.021563,0.545401,0.555072,0.931624,0.938131,0.856447,0.899657,0.876949,0.869179,0.734673
2,Taiwan Credit Card Default,Sigmoid,0.134566,0.002558,0.430391,0.006718,0.011495,0.004005,1.007962,0.221825,0.221200,0.777340,0.557204,0.363076,0.676515,0.472411,0.656856,0.402540
3,Taiwan Credit Card Default,Uncalibrated,0.156585,0.001877,0.490560,0.004278,0.144058,-0.761388,1.250380,0.365091,0.221200,0.777340,0.557204,0.509745,0.573779,0.539777,0.701085,0.420070


,dataset,calibration_method,Delta_Brier,Delta_LogLoss,Delta_ECE,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC
0,Australian Credit Approval,Sigmoid,-0.000037,0.000193,0.004517,0.0,0.0,0.008027,0.000680,-0.001421,-0.002015
1,Australian Credit Approval,Uncalibrated,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,Taiwan Credit Card Default,Sigmoid,-0.022019,-0.060170,-0.132563,0.0,0.0,-0.146669,-0.067366,-0.044230,-0.017530
3,Taiwan Credit Card Default,Uncalibrated,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000


## 9. Final checks

Do not change the calibration method after viewing external results.  
The next stage will optimise decision thresholds using German development data and then freeze the threshold rule for independent replication.


In [11]:

assert len(fold_results) == 2 * 25 * 2
assert len(calibrators) == 2 * 25

expected_predictions = 5 * sum(len(df) for df in datasets.values())
assert len(predictions) == expected_predictions

for column in [
    "uncalibrated_probability",
    "sigmoid_probability",
]:
    assert predictions[column].between(0, 1).all()

assert fold_results["brier_score"].ge(0).all()
assert fold_results["log_loss"].ge(0).all()
assert fold_results["roc_auc"].between(0, 1).all()
assert fold_results["pr_auc"].between(0, 1).all()

configuration = {
    "stage": "Objective 3 Step 10 - frozen sigmoid calibration replication",
    "development_dataset": "German Credit",
    "replication_datasets": list(datasets.keys()),
    "frozen_core_hybrid": (
        "Group-aware Chi2 Top75 + balanced LR/RF/XGB + equal soft voting"
    ),
    "frozen_calibration_method": "Sigmoid",
    "calibrator_training": (
        "5-fold StratifiedGroupKFold inner OOF predictions "
        "inside each outer-training fold"
    ),
    "threshold": 0.50,
    "external_calibration_method_selection": False,
    "threshold_optimization": "deferred",
}

with open(
    OUT_DIR / "step10_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(configuration, f, indent=4)

manifest = sorted(
    [p.name for p in OUT_DIR.iterdir() if p.is_file()]
)

pd.DataFrame(
    {"generated_file": manifest}
).to_csv(
    OUT_DIR / "step10_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 10 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - frozen_sigmoid_replication_summary.csv")
print(" - frozen_sigmoid_replication_delta_summary.csv")
print(" - frozen_sigmoid_replication_fold_results.csv")
print(" - frozen_sigmoid_replication_outer_predictions.csv")
print(" - frozen_sigmoid_calibrator_parameters.csv")


STEP 10 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_sigmoid_calibration_replication

Most important files:
 - frozen_sigmoid_replication_summary.csv
 - frozen_sigmoid_replication_delta_summary.csv
 - frozen_sigmoid_replication_fold_results.csv
 - frozen_sigmoid_replication_outer_predictions.csv
 - frozen_sigmoid_calibrator_parameters.csv
